In [1]:
import json
import csv
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from collections import defaultdict
import os

# Config

In [2]:
DATA_DIR = "./data"
MODELS_DIR = "./models"
GENRE_EMBEDDINGS_PATH = f"{DATA_DIR}/embeddings/genre_embeddings.json"
TAG_EMBEDDINGS_PATH = f"{DATA_DIR}/embeddings/tag_embeddings.json"
SONGS_CSV = f"{DATA_DIR}/csv/songs.csv"
TAGS_CSV = f"{DATA_DIR}/csv/tags.csv"

MODEL_PATH = f"{MODELS_DIR}/vae_model.pt"

EMBED_DIM = 96  # 32 (genre) + 64 (tag)
HIDDEN_DIM = 128
LATENT_DIM = 16
EPOCHS = 200
BATCH_SIZE = 256
LR = 0.001
BETA = 0.005
PATIENCE = 5
MIN_DELTA = 0.0001

# Load Embeddings

In [3]:
with open(GENRE_EMBEDDINGS_PATH) as f: genre_raw = json.load(f)
with open(TAG_EMBEDDINGS_PATH)   as f: tag_raw   = json.load(f)

g_idx = {g: i for i, g in enumerate(genre_raw)}
t_idx = {t: i for i, t in enumerate(tag_raw)}
G_arr = np.array(list(genre_raw.values()), dtype=np.float32)
T_arr = np.array(list(tag_raw.values()),   dtype=np.float32)
GENRE_DIM, TAG_DIM = G_arr.shape[1], T_arr.shape[1]

def read_csv(path):
    with open(path, encoding='utf-8', errors='replace') as f:
        reader = csv.reader(f, delimiter=';')
        headers = [h.strip().strip('"') for h in next(reader)[0].split(',')]
        return [dict(zip(headers, [p.strip().strip('"') for p in row]))
                for row in reader if len(row) == len(headers)]

song_lookup = {}
song_genres = defaultdict(set)
for row in read_csv(SONGS_CSV):
    sid = row['spotify_id']
    song_lookup[sid] = row
    if row['genre_name'] in g_idx:
        song_genres[sid].add(row['genre_name'])

song_tags_full = defaultdict(dict)
for row in read_csv(TAGS_CSV):
    sid, tag = row['song_spotify_id'], row['tag'].lower().strip()
    pop = int(row['popularity']) if row['popularity'].isdigit() else 0
    if tag in t_idx and pop > song_tags_full[sid].get(tag, 0):
        song_tags_full[sid][tag] = pop
song_tags = {sid: sorted(d.items(), key=lambda x: -x[1])[:8]
             for sid, d in song_tags_full.items()}

def song_to_vec(sid):
    g_set = song_genres.get(sid, set())
    if not g_set: return None
    g_mean = np.mean([G_arr[g_idx[g]] for g in g_set], axis=0)
    tags = song_tags.get(sid, [])
    if tags: t_mean = np.mean([T_arr[t_idx[t]] for t, _ in tags], axis=0)
    else:    t_mean = np.zeros(TAG_DIM, dtype=np.float32)
    return np.concatenate([g_mean, t_mean]).astype(np.float32)

song_vectors, song_ids = [], []
for sid in song_genres:
    v = song_to_vec(sid)
    if v is not None:
        song_vectors.append(v)
        song_ids.append(sid)

X = torch.tensor(np.stack(song_vectors), dtype=torch.float32)
print(f"{X.shape[0]:,} songs | embed_dim={X.shape[1]}")

dataset = TensorDataset(X)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

177,079 songs | embed_dim=96


# VAE Class

In [4]:
class VAE(nn.Module):
    def __init__(self, embed_dim, hidden_dim, latent_dim):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim), nn.ReLU()
        )
        self.mu_layer     = nn.Linear(hidden_dim, latent_dim)
        self.logvar_layer = nn.Linear(hidden_dim, latent_dim)
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, embed_dim)
        )

    def encode(self, x):
        h = self.encoder(x)
        return self.mu_layer(h), self.logvar_layer(h)

    def reparameterize(self, mu, logvar):
        logvar = torch.clamp(logvar, min=-4, max=4)
        if self.training:
            return mu + torch.exp(0.5 * logvar) * torch.randn_like(mu)
        return mu

    def decode(self, z):
        return self.decoder(z)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z          = self.reparameterize(mu, logvar)
        return self.decode(z), mu, logvar

def vae_loss(recon, x, mu, logvar, beta=BETA):
    recon_loss = nn.functional.mse_loss(recon, x, reduction='mean')
    kl_loss    = torch.mean(torch.clamp(
        -0.5 * (1 + logvar - mu.pow(2) - logvar.exp()), min=0.01
    ))
    return recon_loss + beta * kl_loss, recon_loss, kl_loss

# Training

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using: {device}")

os.makedirs(MODELS_DIR, exist_ok=True)
model = VAE(EMBED_DIM, HIDDEN_DIM, LATENT_DIM).to(device)

RETRAIN = False   # set True to force retraining

if not RETRAIN and os.path.exists(MODEL_PATH):
    model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
    model.eval()
    print(f"Loaded saved model from {MODEL_PATH}")
else:
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    best_loss, patience_counter = float('inf'), 0

    for epoch in range(EPOCHS):
        model.train()
        total_loss = total_recon = total_kl = 0.0

        for (batch,) in dataloader:
            batch = batch.to(device)
            recon, mu, logvar = model(batch)
            loss, recon_loss, kl_loss = vae_loss(recon, batch, mu, logvar)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            total_recon += recon_loss.item()
            total_kl += kl_loss.item()

        n = len(dataloader)
        avg_loss = total_loss / n
        print(f"Epoch {epoch+1}/{EPOCHS} \u2014 loss={avg_loss:.4f} | recon={total_recon/n:.4f} | kl={total_kl/n:.4f}")

        if avg_loss < best_loss - MIN_DELTA:
            best_loss, patience_counter = avg_loss, 0
            torch.save(model.state_dict(), MODEL_PATH)
        else:
            patience_counter += 1
            if patience_counter >= PATIENCE:
                print("Early stopping.")
                break

    model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
    model.eval()
    print(f"Best loss: {best_loss:.4f} \t saved to {MODEL_PATH}")

Using: cpu
Loaded saved model from ./models/vae_model.pt


# Get latent vectors

In [6]:
model.eval()

with torch.no_grad():
    mu, _ = model.encode(X.to(device))
    Z_all = mu.cpu().numpy()

print(f"Latent vectors: {Z_all.shape}")

Latent vectors: (177079, 16)


In [7]:
song_latents = {song_ids[i]: Z_all[i].tolist() for i in range(len(song_ids))}

with open(f"{DATA_DIR}/embeddings/song_latents.json", "w") as f:
    json.dump(song_latents, f)

print(f"Saved {len(song_latents):,} song latents.")


Saved 177,079 song latents.
